# MonoDETR MobileNetV4 backbone-only ablation

This gate changes only the ResNet50 feature extractor. Depth predictor, transformer, queries, heads, losses, KITTI split, and evaluator remain MonoDETR. Run the 20-epoch gate first; do not authorize a full run until its AP/latency result is reviewed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, shlex, shutil, subprocess, sys
MOBILE_REPO = Path('/content/mobile_adas3d')
MONODETR_REPO = Path('/content/MonoDETR')
MONODETR_COMMIT = '6994b9f512400b258c6edb75f77423beb9c126f2'
DRIVE_DATASET_ROOT = Path('/content/drive/MyDrive/datasets/kitti')
LOCAL_DATASET_ROOT = Path('/content/kitti')
SPLIT_DIR = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
MONODETR_KITTI = Path('/content/monodetr_kitti')
TEACHER_CHECKPOINT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/teachers/monodetr/checkpoint_best.pth')
RUN_DIR = Path('/content/drive/MyDrive/mobile_adas3d_outputs/monodetr_backbone_ablation/mnv4_conv_small_gate20')
BATCH_SIZE = 16
GATE_EPOCHS = 20
LEARNING_RATE = 1e-4
def run(command, cwd=None, env=None):
    command = [str(x) for x in command]; print('+', shlex.join(command), flush=True)
    merged = os.environ.copy(); merged.update(env or {})
    result = subprocess.run(command, cwd=cwd, env=merged)
    if result.returncode: raise RuntimeError(f'Exit {result.returncode}: {shlex.join(command)}')
run(['nvidia-smi'])

In [ ]:
# Repositories and current-Colab compatibility. The MobileADAS3D commit must contain this notebook's companion scripts.
if not MOBILE_REPO.exists(): run(['git', 'clone', 'https://github.com/Ali-RT/mobile_adas3d.git', MOBILE_REPO])
else: run(['git', 'pull', '--ff-only'], cwd=MOBILE_REPO)
if not MONODETR_REPO.exists(): run(['git', 'clone', 'https://github.com/ZrrSkywalker/MonoDETR.git', MONODETR_REPO])
run(['git', 'fetch', '--all'], cwd=MONODETR_REPO)
run(['git', 'checkout', MONODETR_COMMIT], cwd=MONODETR_REPO)
run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown', 'pyyaml', 'scipy', 'opencv-python-headless', 'numba', 'scikit-image', 'tqdm', 'ninja', 'timm==1.0.20'])
run([sys.executable, 'scripts/patch_monodetr_colab_compat.py', '--monodetr-repo', MONODETR_REPO], cwd=MOBILE_REPO)
run([sys.executable, 'scripts/patch_monodetr_mobilenetv4.py', '--monodetr-repo', MONODETR_REPO], cwd=MOBILE_REPO)
ops = MONODETR_REPO / 'lib/models/monodetr/ops'
shutil.rmtree(ops / 'build', ignore_errors=True)
run([sys.executable, 'setup.py', 'build', 'install'], cwd=ops, env={'MAX_JOBS': '2'})
run([sys.executable, '-c', 'import torch, timm, MultiScaleDeformableAttention; from lib.models.monodetr import build_monodetr; print(torch.__version__, timm.__version__, torch.cuda.get_device_name(0))'], cwd=MONODETR_REPO)

In [ ]:
# Create a zero-copy KITTI view using either the fast local stage or Google Drive aliases.
def resolve(root, names):
    for name in names:
        path = root / name
        if path.is_dir(): return path
    return None
sources = {}
for key, names in {'image_2':['training/image_2','training/image_02'], 'label_2':['training/label_2','training/label_02'], 'calib':['training/calib']}.items():
    sources[key] = resolve(LOCAL_DATASET_ROOT, names) or resolve(DRIVE_DATASET_ROOT, names)
if any(path is None for path in sources.values()): raise FileNotFoundError(f'Missing KITTI sources: {sources}')
(MONODETR_KITTI/'training').mkdir(parents=True, exist_ok=True); (MONODETR_KITTI/'ImageSets').mkdir(parents=True, exist_ok=True)
for name, target in sources.items():
    link = MONODETR_KITTI/'training'/name
    if link.is_symlink() and link.resolve() == target.resolve(): continue
    if link.exists() or link.is_symlink(): raise RuntimeError(f'Refusing to replace {link}')
    link.symlink_to(target, target_is_directory=True)
for split in ('train','val'): shutil.copy2(SPLIT_DIR/f'{split}.txt', MONODETR_KITTI/'ImageSets'/f'{split}.txt')
assert len((MONODETR_KITTI/'ImageSets/train.txt').read_text().splitlines()) == 3712
assert len((MONODETR_KITTI/'ImageSets/val.txt').read_text().splitlines()) == 3769
print('KITTI view ready:', MONODETR_KITTI)

In [ ]:
# Prepare strict-load initialization: ImageNet MobileNetV4 plus compatible official MonoDETR downstream tensors.
if not TEACHER_CHECKPOINT.is_file(): raise FileNotFoundError('Run/download the validated official MonoDETR teacher checkpoint first: ' + str(TEACHER_CHECKPOINT))
run([sys.executable, 'scripts/prepare_monodetr_mnv4_backbone_experiment.py', '--monodetr-repo', MONODETR_REPO, '--dataset-root', MONODETR_KITTI, '--official-checkpoint', TEACHER_CHECKPOINT, '--output-dir', RUN_DIR, '--run-name', RUN_DIR.name, '--max-epochs', GATE_EPOCHS, '--save-frequency', 5, '--batch-size', BATCH_SIZE, '--learning-rate', LEARNING_RATE], cwd=MOBILE_REPO)
CONFIG = MONODETR_REPO/'configs/monodetr_mnv4_conv_small_backbone_gate.yaml'
print(CONFIG.read_text())

## Gate training

This is the real training loop. It writes checkpoints and logs directly to Drive. If batch 16 runs out of memory, restart the runtime and change `BATCH_SIZE` to 8; record that change because it weakens strict comparability.

In [ ]:
run([sys.executable, '-u', 'tools/train_val.py', '--config', CONFIG], cwd=MONODETR_REPO)

In [ ]:
# Inspect durable artifacts after training. Share experiment_manifest.json, train log, checkpoint paths, and official KITTI AP output.
print((RUN_DIR/'experiment_manifest.json').read_text())
print('Run artifacts:')
for path in sorted(RUN_DIR.rglob('*')):
    if path.is_file(): print(path.relative_to(RUN_DIR), round(path.stat().st_size/1e6, 2), 'MB')